In [1]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv

C:\Users\SUBHAM\AppData\Local\Temp\ipykernel_19676\2235675510.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\COURSES\AIML\GEN_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [12]:

# this part is only ai generated part

from langchain_community.document_loaders import DirectoryLoader, PDFPlumberLoader
from langchain_core.documents import Document

book_folder = "./books"

loader = DirectoryLoader(
    book_folder,
    glob="*.pdf",
    loader_cls=PDFPlumberLoader,
    show_progress=True
)

pages = loader.load()

documents = []

# Group pages by PDF
books = {}

for page in pages:
    source = page.metadata["source"]
    books.setdefault(source, []).append(page)

# Create one Document for each PDF
for source, book_pages in sorted(books.items()):
    full_text = "\n\n".join(
        page.page_content for page in book_pages
    )

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": source,
                "file_name": source.split("\\")[-1],
                "total_pages": len(book_pages)
            }
        )
    )

print("Total documents:", len(documents))



100%|██████████| 3/3 [00:42<00:00, 14.04s/it]

Total documents: 3


In [13]:
documents

[Document(metadata={'source': 'books\\book1.pdf', 'file_name': 'book1.pdf', 'total_pages': 326}, page_content='[ 1 ]\n\n\nBuilding Machine Learning\nSystems with Python\nSecond Edition\nGet more from your data through creating practical\nmachine learning systems with Python\nLuis Pedro Coelho\nWilli Richert\nBIRMINGHAM - MUMBAI\n\n\nBuilding Machine Learning Systems with Python\nSecond Edition\nCopyright © 2015 Packt Publishing\nAll rights reserved. No part of this book may be reproduced, stored in a retrieval\nsystem, or transmitted in any form or by any means, without the prior written\npermission of the publisher, except in the case of brief quotations embedded in\ncritical articles or reviews.\nEvery effort has been made in the preparation of this book to ensure the accuracy\nof the information presented. However, the information contained in this book is\nsold without warranty, either express or implied. Neither the authors, nor Packt\nPublishing, and its dealers and distributors 

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [18]:
chunks=splitter.split_documents(documents)

In [19]:
len(chunks)

2789

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

emb_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5620.93it/s]


In [21]:
from langchain_community.vectorstores import FAISS

vector_store=FAISS.from_documents(
    chunks,
    emb_model,
)

In [22]:
retriver=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

In [23]:
retriver.invoke("what is linear regression?")

[Document(id='19f7e9cc-2387-4590-b409-7b4c3d149f03', metadata={'source': 'books\\book3.pdf', 'file_name': 'book3.pdf', 'total_pages': 392}, page_content='Linear Models\nLinear models are a class of models that are widely used in practice and have been\nstudied extensively in the last few decades, with roots going back over a hundred\nyears. Linear models make a prediction using a linear function of the input features,\nwhich we will explain shortly.\nLinear models for regression\nFor regression, the general prediction formula for a linear model looks as follows:\nŷ = w[0] * x[0] + w[1] * x[1] + ... + w[p] * x[p] + b\nHere, x[0] to x[p] denotes the features (in this example, the number of features is p)\nof a single data point, w and b are parameters of the model that are learned, and ŷ is\nthe prediction the model makes. For a dataset with a single feature, this is:\nŷ = w[0] * x[0] + b\nwhich you might remember from high school mathematics as the equation for a line.\nHere, w[0] is th

In [24]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate(
    template="""you are a helpful machine learning ai assistsnt
    please answer the question \n {question}
    from the following content \n {content}
    """,
    input_variables=["question","content"]
)

In [25]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-120b")

In [26]:
def format_docs(retrieved_docs):
    context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
    )
    return context

In [28]:
from langchain_core.runnables import RunnablePassthrough,RunnableParallel,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

parser=StrOutputParser()

In [29]:
chain=({
     "question":RunnablePassthrough(),
     "content":retriver | RunnableLambda(format_docs)
} | prompt | model | parser )


In [33]:
query="what is linear regression in short?"


result=chain.invoke(query)

result

'**Linear regression (ordinary least squares)** is a simple linear model for regression that predicts the target\u202f\\( \\hat{y} \\) as a weighted sum of the input features plus a bias term:\n\n\\[\n\\hat{y}= w_0 x_0 + w_1 x_1 + \\dots + w_p x_p + b\n\\]\n\nThe parameters \\(w\\) (weights) and \\(b\\) (intercept) are learned from the training data by **minimizing the mean‑squared error** between the predictions \\(\\hat{y}\\) and the true targets \\(y\\). In other words, it finds the line (or hyper‑plane in higher dimensions) that best fits the data in the least‑squares sense. This is the classic, computationally fast method often introduced in high‑school mathematics.'